# Week 2 Goal

Compare different structured extraction strategies and measure reliability:

- Extraction Strategies
- Naive JSON prompt
- Schema-guided prompt
- Strict + validation-aware prompt



---



## Install Dependencies

In [1]:
# Import pydantic
!pip install openai pydantic python-dotenv

In [4]:
#Import anthropic
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 12.0 MB/s eta 0:00:00


## Libraries

In [40]:
import anthropic

client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)

## Model Restrictions

- For budget limiting tokens to 200
- Good enough for structured extraction
- No randomness, Temp = 0

In [38]:
# Set Model to 200
model="claude-sonnet-4-6"

In [39]:
# Limit Tokens

def call_llm(prompt, model="claude-sonnet-4-6"):
    import time

    start = time.time()

    response = client.messages.create(
        model=model,
        max_tokens=200,
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    latency = time.time() - start

    output = response.content[0].text

    return {
        "output": output,
        "latency": latency,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

In [41]:
# Test Model
call_llm("Return JSON: {hello: world}")

{'output': '```json\n{\n  "hello": "world"\n}\n```',
 'latency': 0.9579951763153076,
 'input_tokens': 15,
 'output_tokens': 19}

**Insight**

- Passes formatting requirement (valid JSON)
- No hallucinations or malformed structure
- No meaningful task completion beyond echoing input
- No reasoning, extraction, or transformation demonstrated



---



## Use Case: Fintech Customer Support Ticket Intake

- This mirrors a production triage system where an LLM extracts structured risk signals from messy user messages.

In [42]:
# Input Examples to Test

examples = [
"""
Hi, I noticed two charges for $842 from merchants I do not recognize.
Card ending 4412 may be compromised. I am currently traveling in Mexico.
Please freeze my card and dispute both transactions.
""",

"""
I missed my loan payment due March 1 because I lost my job.
Can I request hardship assistance and avoid collections?
""",

"""
I received an SMS saying my account is locked and asking for my password.
Is this fraud? The message came from +1 800 555 1212.
"""
]

## Target Schema

In [43]:
from pydantic import BaseModel
from typing import List, Optional

class SupportExtraction(BaseModel):
    issue_type: str
    urgency_level: str
    fraud_flag: bool
    monetary_amount: Optional[float]
    requested_actions: List[str]
    sentiment: str



---



## 3 Method Approach

**Method 1: Naive Prompting (Baseline)**
- Represents how many teams start in production: simple instructions asking the model to return JSON. It establishes a baseline for comparison and reveals common failures such as malformed JSON, missing fields, or hallucinated values.

**Method 2: Schema Guided Prompting (Structure Control)**
- Adds an explicit schema to test whether stronger formatting constraints improve compliance and field level accuracy.

**Method 3: Strict Validation Aware Prompting (Production Hardening)**
- Introduces business rules such as allowed labels, null handling, and “never invent values,” which simulates guardrails used in production systems.

In [45]:
# Method 1: Naive Prompt

prompt = f'''
Extract as JSON:

Fields:
issue_type
urgency_level
fraud_flag
monetary_amount
requested_actions
sentiment

Text:
{examples}
'''

In [46]:
# Method 2: Schema Guided

schema_prompt = f'''
Return only valid JSON matching:

{{
"issue_type": string,
"urgency_level": string,
"fraud_flag": boolean,
"monetary_amount": number or null,
"requested_actions": [string],
"sentiment": string
}}

Input:
{examples}
'''

In [47]:
# Method 3: Strict Validation Aware

strict_prompt = f'''
Return valid JSON only.

Rules:
- Use null if missing
- Never invent values
- requested_actions must be a list
- urgency_level must be low medium or high
- issue_type must be one of:
fraud, hardship, phishing, dispute

Input:
{examples}
'''



---



## Evaluation

In [48]:
import json
import pandas as pd

results=[]

for text in examples:
    for method,prompt in {
        "naive": prompt,
        "schema": schema_prompt,
        "strict": strict_prompt
    }.items():

        r = call_llm(prompt)

        try:
            parsed=json.loads(r["output"])
            valid_json=True
        except:
            valid_json=False
            parsed={}

        results.append({
            "method":method,
            "valid_json":valid_json,
            "latency":r["latency"],
            "input_tokens":r["input_tokens"],
            "output_tokens":r["output_tokens"]
        })

df=pd.DataFrame(results)
df.groupby("method").mean()

,valid_json,latency,input_tokens,output_tokens
method,,,,
naive,0.0,5.348254,186.666667,200.0
schema,0.0,4.296457,195.000000,200.0
strict,0.0,3.495099,197.000000,200.0


**Insight**

Across all prompting strategies, JSON validity was 0%, indicating that the primary failure mode is output formatting rather than semantic extraction. This suggests that prompt engineering alone is insufficient and that structured output enforcement or post-processing is required for production reliability.